# Notebook 38 — Global Posterior Qualification Gate

**Route B** synthetic-domain global posterior qualification for Part II.

- Primary track: **7D**; sensitivity: **8D**
- E2_ref frozen aliases: 7D `member_4`, 8D `member_3` (verified from NB36 Phase 4)
- This notebook's default path is **`development` only**
- **Sealed confirmation is not executed here** — even after Dev PASS the status remains `PENDING_SEALED_CONFIRMATION`
- Kernel: `neurolib`


## 1. Setup and claim ceiling

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Project-root discovery (notebook lives in S4_sbi/notebooks/)
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
else:
    # Fallback: walk up until S4_sbi/src exists
    PROJECT_ROOT = NOTEBOOK_DIR
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (parent / "S4_sbi" / "src" / "sleep_sbi").is_dir():
            PROJECT_ROOT = parent
            break

SRC = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sleep_sbi.figure10_notebook38_global_gate import (
    EIGHT_D_SEALED_REQUIRED_FOR_NB41,
    N38_ROOT,
    assert_path_not_sealed_case_arrays,
    aggregate_support_qa,
    bank_overlap_audit,
    build_track_decision,
    checkpoint_reload_audit,
    compose_final_qualification,
    compute_marginal_coverage,
    ensure_dirs,
    freeze_protocol_lock,
    generation_and_provenance_manifest,
    ingest_legacy_sealed_markers,
    load_sealed_state_registry,
    public_sealed_status,
    authorize_track_in_registry,
    resolve_e2_ref_aliases,
    run_global_support_safe_diagnostics,
    run_lc2st_development,
    sealed_eligibility_for_track,
    statistical_spec_fingerprint,
    write_artifact_hash_inventory,
    write_candidate_ledger,
    write_diagnostic_figures,
    write_figure_inventory,
    json_safe,
    utc_now,
)
from sleep_sbi.figure10_protocol import atomic_json

paths = ensure_dirs()
display(Markdown(
    f"**PROJECT_ROOT** = `{PROJECT_ROOT}`  \n"
    f"**N38 artifacts** = `{N38_ROOT}`  \n"
    f"**eight_d_sealed_required_for_nb41** = `{EIGHT_D_SEALED_REQUIRED_FOR_NB41}` (default False → 8D sealed DENIED)"
))


**PROJECT_ROOT** = `D:\Year3_Mao_Projects\sleep_loop`  
**N38 artifacts** = `D:\Year3_Mao_Projects\sleep_loop\S4_sbi\results\figure10_8d_7d\artifacts\notebook_38`  
**eight_d_sealed_required_for_nb41** = `False` (default False → 8D sealed DENIED)

## 2. Settled facts from Notebooks 33–37

In [2]:
settled = {
    "claim_route": "B",
    "statistical_role": "working_prior_box",
    "primary_track": "7d",
    "sensitivity_track": "8d",
    "nb37_status": "CONDITIONAL_GO",
    "branch": "Branch1_reuse_existing_estimator_subject_to_nb38_global_gate",
    "powered_1024_exposure": "FULLY_ANALYZED_IN_NB35_NB36",
    "pilot_split": None,
    "retrain": False,
    "new_training_or_calibration_sims_in_nb38": False,
    "forbidden": [
        "real_eeg_personalized_posterior",
        "prior_narrowing",
        "calibration_hacks",
        "sealed_as_selection_bank",
        "lc2st_as_sealed_independent_evidence",
    ],
}
display(pd.DataFrame([{"key": k, "value": str(v)} for k, v in settled.items()]))


,key,value
0,claim_route,B
1,statistical_role,working_prior_box
2,primary_track,7d
3,sensitivity_track,8d
4,nb37_status,CONDITIONAL_GO
5,branch,Branch1_reuse_existing_estimator_subject_to_nb...
6,powered_1024_exposure,FULLY_ANALYZED_IN_NB35_NB36
7,pilot_split,None
8,retrain,False
9,new_training_or_calibration_sims_in_nb38,False


## 3. Protocol lock prerequisites — legacy sealed markers (metadata only)

In [3]:
legacy = ingest_legacy_sealed_markers()
display(Markdown(f"**legacy_status** = `{legacy.get('legacy_status')}`"))
if legacy.get("legacy_status") == "LEGACY_BLOCKED":
    raise RuntimeError(legacy.get("block_reason"))
# Negative test: case-level sealed arrays must be denied
try:
    assert_path_not_sealed_case_arrays(
        PROJECT_ROOT
        / "S4_sbi/results/figure10_8d_7d/artifacts/notebook_36/sealed/"
        "sealed_confirm_512/7d/shards/shard_0000000_0000064.npz"
    )
    raise RuntimeError("expected SealedAccessError")
except Exception as exc:
    display(Markdown(f"Sealed case-array negative test: `{type(exc).__name__}: {exc}`"))


**legacy_status** = `UNOPENED`

Sealed case-array negative test: `SealedAccessError: sealed case-level path forbidden: D:/Year3_Mao_Projects/sleep_loop/S4_sbi/results/figure10_8d_7d/artifacts/notebook_36/sealed/sealed_confirm_512/7d/shards/shard_0000000_0000064.npz`

## 4. Input fingerprint, provenance, overlap, checkpoints

In [4]:
fingerprint = statistical_spec_fingerprint()
provenance = generation_and_provenance_manifest()
overlap = bank_overlap_audit()
checkpoints = checkpoint_reload_audit()
e2_ref = resolve_e2_ref_aliases()

display(Markdown(
    f"**Situation** = `{fingerprint['situation']}`  ·  "
    f"**fingerprints_equal** = `{fingerprint['fingerprints_equal']}`  ·  "
    f"**correction** = `{fingerprint['correction']}`"
))
display(Markdown(
    f"**E2_ref** 7D=`{e2_ref['7d']}` · 8D=`{e2_ref['8d']}` · "
    f"verified={e2_ref['verified_from_nb36_phase4']}"
))
display(Markdown(
    f"**Checkpoints all_ok** = `{checkpoints.get('all_ok')}`  ·  "
    f"**paired_designs** = `{overlap.get('paired_designs')}`"
))


**Situation** = `A`  ·  **fingerprints_equal** = `True`  ·  **correction** = `NO_PRIOR_PROPOSAL_CORRECTION_REQUIRED`

**E2_ref** 7D=`member_4` · 8D=`member_3` · verified=True

**Checkpoints all_ok** = `True`  ·  **paired_designs** = `32768`

## 5. Freeze protocol lock and candidate ledger (before diagnostics)

In [5]:
if fingerprint.get("situation") != "A":
    raise RuntimeError("Situation B — fail closed; no diagnostics")
if not checkpoints.get("all_ok"):
    raise RuntimeError("checkpoint reload failed — fail closed")

lock = freeze_protocol_lock(
    e2_ref=e2_ref,
    fingerprint=fingerprint,
    legacy=legacy,
    overlap=overlap,
)
ledger = write_candidate_ledger(e2_ref)
display(Markdown(f"Protocol frozen at `{paths['json'] / 'protocol_lock.json'}`"))
display(ledger)


Protocol frozen at `D:\Year3_Mao_Projects\sleep_loop\S4_sbi\results\figure10_8d_7d\artifacts\notebook_38\json\protocol_lock.json`

,track,candidate_id,estimator,role,gates,alias_of
0,7d,E0,ensemble,negative_control_scorecard,False,None
1,7d,E1,member_1,full_dev_scorecard,False,None
2,7d,E1,member_2,full_dev_scorecard,False,None
3,7d,E1,member_3,full_dev_scorecard,False,None
4,7d,E2_ref,member_4,sole_dev_gate_and_sealed_evaluatee,True,E1_member_4
5,7d,E2_ref,member_4,alias_into_E1_no_second_sampling_job,True,member_4
6,7d,E1,member_5,full_dev_scorecard,False,None
7,8d,E0,ensemble,negative_control_scorecard,False,None
8,8d,E1,member_1,full_dev_scorecard,False,None
9,8d,E1,member_2,full_dev_scorecard,False,None


## 6. Development exposure note

In [6]:
exposure = {
    "powered_1024": "FULLY_ANALYZED_IN_NB35_NB36",
    "role": "Development-only_internal_audit",
    "pristine_holdout": False,
    "pilot_256_768": False,
    "e2_ref_selection": e2_ref["label"],
}
display(pd.Series(exposure))


powered_1024                              FULLY_ANALYZED_IN_NB35_NB36
role                                  Development-only_internal_audit
pristine_holdout                                                False
pilot_256_768                                                   False
e2_ref_selection    frozen_from_nb36_joint_ks_on_exposed_powered_1024
dtype: object

## 7. Support-safe global diagnostics — 7D primary

In [7]:
consolidated_7d = run_global_support_safe_diagnostics("7d")
cov_7d = compute_marginal_coverage("7d", consolidated_7d)
qa_7d = aggregate_support_qa("7d", e2_ref["7d"])
figs_7d = write_diagnostic_figures("7d", consolidated_7d)
display(Markdown(f"**7D E2_ref support hard_fail** = `{qa_7d.get('hard_fail')}`"))
display(pd.read_csv(paths["csv"] / "7d_global_summary.csv"))
display(pd.read_csv(paths["csv"] / "7d_marginal_sbc_summary.csv").head(20))


**7D E2_ref support hard_fail** = `True`

,dataset_id,track,scale,estimator,cases,joint_rank_ks_statistic,joint_rank_ks_pvalue,mean_posterior_median_abs_error,prior_median_abs_error,recovery_improvement_fraction,median_normalized_posterior_sd
0,powered_1024_support_safe_n38,7d,32768,ensemble,1024,0.210175,3.661816e-40,0.124159,0.25,0.503362,0.564506
1,powered_1024_support_safe_n38,7d,32768,member_1,1024,0.047247,2.001455e-02,0.126529,0.25,0.493883,0.566074
2,powered_1024_support_safe_n38,7d,32768,member_2,1024,0.029211,3.400071e-01,0.121019,0.25,0.515925,0.566391
3,powered_1024_support_safe_n38,7d,32768,member_3,1024,0.034184,1.784260e-01,0.126918,0.25,0.492330,0.557905
4,powered_1024_support_safe_n38,7d,32768,member_4,1024,0.022653,6.608302e-01,0.126807,0.25,0.492771,0.562301
5,powered_1024_support_safe_n38,7d,32768,member_5,1024,0.022847,6.503495e-01,0.127282,0.25,0.490872,0.601698


,dataset_id,track,scale,estimator,parameter,ks_statistic,ks_pvalue,rank_mean,rank_variance,bias_mean,normalized_posterior_sd_median,holm_clear_issue
0,powered_1024_support_safe_n38,7d,32768,ensemble,mue,0.072554,3.921078e-05,0.518349,0.100833,-0.011407,0.549963,True
1,powered_1024_support_safe_n38,7d,32768,ensemble,mui,0.188984,1.696724e-32,0.457762,0.038685,0.001288,0.031568,True
2,powered_1024_support_safe_n38,7d,32768,ensemble,b,0.024645,5.542705e-01,0.507084,0.087070,-0.001043,0.361447,False
3,powered_1024_support_safe_n38,7d,32768,ensemble,tauA,0.068928,1.126437e-04,0.519765,0.100575,-0.010782,0.725430,True
4,powered_1024_support_safe_n38,7d,32768,ensemble,g_LK,0.040103,7.223147e-02,0.503977,0.091981,0.015711,0.564506,False
5,powered_1024_support_safe_n38,7d,32768,ensemble,g_h,0.032406,2.274825e-01,0.506863,0.084801,0.016365,0.669296,False
6,powered_1024_support_safe_n38,7d,32768,ensemble,c_th2ctx,0.042920,4.465066e-02,0.496469,0.094523,-0.046276,0.629094,False
7,powered_1024_support_safe_n38,7d,32768,member_1,mue,0.069074,1.080607e-04,0.521709,0.098854,-0.014362,0.566074,True
8,powered_1024_support_safe_n38,7d,32768,member_1,mui,0.257510,2.275880e-60,0.632077,0.048191,-0.003386,0.026867,True
9,powered_1024_support_safe_n38,7d,32768,member_1,b,0.028318,3.772402e-01,0.494036,0.089182,0.003522,0.357281,False


## 8. Support-safe global diagnostics — 8D sensitivity

In [8]:
consolidated_8d = run_global_support_safe_diagnostics("8d")
cov_8d = compute_marginal_coverage("8d", consolidated_8d)
qa_8d = aggregate_support_qa("8d", e2_ref["8d"])
figs_8d = write_diagnostic_figures("8d", consolidated_8d)
display(Markdown(f"**8D E2_ref support hard_fail** = `{qa_8d.get('hard_fail')}`"))
display(pd.read_csv(paths["csv"] / "8d_global_summary.csv"))


**8D E2_ref support hard_fail** = `True`

,dataset_id,track,scale,estimator,cases,joint_rank_ks_statistic,joint_rank_ks_pvalue,mean_posterior_median_abs_error,prior_median_abs_error,recovery_improvement_fraction,median_normalized_posterior_sd
0,powered_1024_support_safe_n38,8d,32768,ensemble,1023,0.220570,3.437686e-44,0.142137,0.24986,0.431136,0.637083
1,powered_1024_support_safe_n38,8d,32768,member_1,1023,0.043639,3.943413e-02,0.144724,0.24986,0.420779,0.635843
2,powered_1024_support_safe_n38,8d,32768,member_2,1023,0.061825,7.665380e-04,0.142324,0.24986,0.430387,0.631935
3,powered_1024_support_safe_n38,8d,32768,member_3,1023,0.025131,5.296616e-01,0.139516,0.24986,0.441626,0.589382
4,powered_1024_support_safe_n38,8d,32768,member_4,1023,0.046421,2.356535e-02,0.146366,0.24986,0.414208,0.640479
5,powered_1024_support_safe_n38,8d,32768,member_5,1023,0.027677,4.061380e-01,0.144488,0.24986,0.421725,0.646589


## 9. L-C2ST (Development only — N36 frozen protocol; not Sealed evidence)

In [9]:
lc2st_summary = run_lc2st_development(skip=False)
lc2st_csv = paths["csv"] / "lc2st_results.csv"
if lc2st_csv.is_file():
    display(pd.read_csv(lc2st_csv))
else:
    display(Markdown(f"L-C2ST status: `{lc2st_summary.get('status')}`"))


n38 lc2st 7d/ensemble: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/7d/ensemble/lc2st_support_safe_result.json)


n38 lc2st 7d/member_1: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/7d/member_1/lc2st_support_safe_result.json)


n38 lc2st 7d/member_2: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/7d/member_2/lc2st_support_safe_result.json)


n38 lc2st 7d/member_3: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/7d/member_3/lc2st_support_safe_result.json)


n38 lc2st 7d/member_4: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/7d/member_4/lc2st_support_safe_result.json)


...

n38 lc2st 8d/member_2: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/8d/member_2/lc2st_support_safe_result.json)


n38 lc2st 8d/member_3: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/8d/member_3/lc2st_support_safe_result.json)


n38 lc2st 8d/member_4: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/8d/member_4/lc2st_support_safe_result.json)


n38 lc2st 8d/member_5: cached (S4_sbi/results/figure10_8d_7d/artifacts/notebook_38/samples/lc2st/8d/member_5/lc2st_support_safe_result.json)


,created_utc,track,estimator,sampling,calibration_cases,calibration_cases_available,extreme_leakage_rows_dropped,local_posterior_samples,num_trials_null,z_score,...,p_value,reject_alpha_0p05,archived_p_value_clipped,archived_reject_clipped,archived_runtime_s,classifier_seed,runtime_s,boundary_rows_dropped,protocol,development_only
0,2026-08-06T08:03:14.297619+00:00,7d,ensemble,support_safe_rejection,19999,20000,1,10000,100,False,...,0.0,True,0.0,False,2632.030922,10135001,3050.622228,NaN,NaN,NaN
1,2026-08-06T08:48:30.519353+00:00,7d,member_1,support_safe_rejection,19998,20000,2,10000,100,False,...,0.0,True,0.0,False,2780.801623,10136010,2716.184006,NaN,NaN,NaN
2,2026-08-09T04:57:02.468147+00:00,7d,member_2,support_safe_rejection,19997,20000,2,10000,100,0,...,0.0,1,NaN,NaN,NaN,10137019,2856.758761,1.0,n36_phase2b_frozen_mlp_zscore_false_null100,1.0
3,2026-08-09T05:46:53.005491+00:00,7d,member_3,support_safe_rejection,19999,20000,1,10000,100,0,...,0.0,1,NaN,NaN,NaN,10138028,2990.511721,0.0,n36_phase2b_frozen_mlp_zscore_false_null100,1.0
4,2026-08-09T06:36:02.405400+00:00,7d,member_4,support_safe_rejection,19997,20000,3,10000,100,0,...,0.0,1,NaN,NaN,NaN,10139037,2949.363148,0.0,n36_phase2b_frozen_mlp_zscore_false_null100,1.0
5,2026-08-09T07:24:05.274454+00:00,7d,member_5,support_safe_rejection,19997,20000,2,10000,100,0,...,0.0,1,NaN,NaN,NaN,10140046,2882.833463,1.0,n36_phase2b_frozen_mlp_zscore_false_null100,1.0
6,2026-08-06T06:17:31.057393+00:00,8d,ensemble,support_safe_rejection,19999,20000,1,10000,100,False,...,0.0,True,0.0,False,3400.972876,10134001,3435.540822,NaN,NaN,NaN
7,2026-08-06T07:12:23.240491+00:00,8d,member_1,support_safe_rejection,19993,20000,7,10000,100,False,...,0.0,True,0.0,False,3334.950489,10135010,3292.139693,NaN,NaN,NaN
8,2026-08-09T08:22:06.282948+00:00,8d,member_2,support_safe_rejection,19993,20000,5,10000,100,0,...,0.0,1,NaN,NaN,NaN,10136019,3480.566045,2.0,n36_phase2b_frozen_mlp_zscore_false_null100,1.0
9,2026-08-09T09:23:46.860192+00:00,8d,member_3,support_safe_rejection,19998,20000,1,10000,100,0,...,0.0,1,NaN,NaN,NaN,10137028,3700.535643,1.0,n36_phase2b_frozen_mlp_zscore_false_null100,1.0


## 10. Candidate scorecard and E2_ref atoms

In [10]:
track_decisions = {}
for track in ("7d", "8d"):
    track_decisions[track] = build_track_decision(
        track,
        e2_ref=e2_ref,
        fingerprint=fingerprint,
        lc2st_summary=lc2st_summary,
        integrity_ok=bool(checkpoints.get("all_ok")),
    )
scorecard = pd.DataFrame(
    [
        {
            "track": t,
            "e2_ref": d["e2_ref"],
            "development_status": d["development_status"],
            **{f"atom_{k}": v for k, v in d["atoms"].items() if not isinstance(v, list)},
        }
        for t, d in track_decisions.items()
    ]
)
display(scorecard)
for track, d in track_decisions.items():
    display(Markdown(f"### {track} atoms"))
    display(pd.Series(d["atoms"]))


,track,e2_ref,development_status,atom_joint,atom_sbc,atom_coverage,atom_lc2st,atom_joint_p,atom_sbc_clear_issues
0,7d,member_4,DEV_FAIL,J_pass,S_fail,C_fail,L_fail,0.660830,3
1,8d,member_3,DEV_FAIL,J_pass,S_fail,C_lim,L_fail,0.529662,3


### 7d atoms

joint                J_pass
sbc                  S_fail
coverage             C_fail
lc2st                L_fail
joint_p             0.66083
sbc_clear_issues          3
dtype: object

### 8d atoms

joint                 J_pass
sbc                   S_fail
coverage               C_lim
lc2st                 L_fail
joint_p             0.529662
sbc_clear_issues           3
dtype: object

## 11. Development decision (stop before Sealed)

In [11]:
seven_d_dev = track_decisions["7d"]["development_status"]
eight_d_dev = track_decisions["8d"]["development_status"]
seven_elig = sealed_eligibility_for_track("7d", seven_d_dev, seven_d_dev)
eight_elig = sealed_eligibility_for_track("8d", eight_d_dev, seven_d_dev)

registry = load_sealed_state_registry()
authorization_freeze = None
if seven_elig == "AUTHORIZED":
    authorize_track_in_registry("7d")
if eight_elig == "AUTHORIZED":
    authorize_track_in_registry("8d")
registry = load_sealed_state_registry()
if seven_elig == "AUTHORIZED" or eight_elig == "AUTHORIZED":
    authorization_freeze = {
        "created_utc": utc_now(),
        "seven_d_sealed_eligibility": seven_elig,
        "eight_d_sealed_eligibility": eight_elig,
        "e2_ref": dict(e2_ref),
        "pending": True,
        "note": "Await explicit sealed_confirm in a separate authorized session.",
    }
    atomic_json(
        paths["json"] / "sealed_authorization_freeze_manifest.json",
        json_safe(authorization_freeze),
    )

seven_sealed = public_sealed_status("7d", registry)
eight_sealed = public_sealed_status("8d", registry)
if seven_elig == "AUTHORIZED" and seven_sealed == "NOT_OPENED":
    seven_sealed = "AWAITING_CONFIRMATION"
if eight_elig == "AUTHORIZED" and eight_sealed == "NOT_OPENED":
    eight_sealed = "AWAITING_CONFIRMATION"

finals = compose_final_qualification(
    seven_d_dev=seven_d_dev,
    seven_d_sealed_status=seven_sealed,
    eight_d_dev=eight_d_dev,
    eight_d_sealed_status=eight_sealed,
)

decision = json_safe({
    "created_utc": utc_now(),
    "stage": "development",
    **finals,
    "seven_d_development_status": seven_d_dev,
    "eight_d_development_status": eight_d_dev,
    "seven_d_sealed_eligibility": seven_elig,
    "eight_d_sealed_eligibility": eight_elig,
    "seven_d_sealed_status": seven_sealed,
    "eight_d_sealed_status": eight_sealed,
    "eight_d_sealed_required_for_nb41": bool(EIGHT_D_SEALED_REQUIRED_FOR_NB41),
    "track_decisions": track_decisions,
    "e2_ref": dict(e2_ref),
    "authorization_freeze_written": authorization_freeze is not None,
    "sealed_opened": False,
    "sealed_consumption_marker_created": False,
})
atomic_json(paths["json"] / "development_decision.json", decision)
atomic_json(paths["json"] / "decision_gate.json", decision)
atomic_json(
    paths["json"] / "notebook39_handoff.json",
    json_safe({
        "created_utc": utc_now(),
        "nb39_handoff_mode": finals["nb39_handoff_mode"],
        "final_qualification_status": finals["final_qualification_status"],
        "blocked_pending_sealed": finals["final_qualification_status"]
        == "PENDING_SEALED_CONFIRMATION",
        "seven_d_sealed_eligibility": seven_elig,
        "eight_d_sealed_eligibility": eight_elig,
    }),
)
write_figure_inventory(figs_7d + figs_8d)
write_artifact_hash_inventory()

display(Markdown("### Development decision"))
display(pd.Series({
    "seven_d_development_status": seven_d_dev,
    "eight_d_development_status": eight_d_dev,
    "seven_d_sealed_eligibility": seven_elig,
    "eight_d_sealed_eligibility": eight_elig,
    "seven_d_sealed_status": seven_sealed,
    "eight_d_sealed_status": eight_sealed,
    "workflow_status": finals["workflow_status"],
    "final_qualification_status": finals["final_qualification_status"],
    "nb39_handoff_mode": finals["nb39_handoff_mode"],
    "sealed_opened": False,
}))
assert decision["sealed_opened"] is False
assert decision["sealed_consumption_marker_created"] is False
# PASS_GLOBAL is forbidden until sealed; pending or limited/no_go only
assert decision["final_qualification_status"] in {
    "PENDING_SEALED_CONFIRMATION",
    "LIMITED_GO",
    "NO_GO",
}
display(Markdown(
    "**STOP:** Development complete. Do not open Sealed without explicit authorization."
))


### Development decision

seven_d_development_status                  DEV_FAIL
eight_d_development_status                  DEV_FAIL
seven_d_sealed_eligibility                    DENIED
eight_d_sealed_eligibility                    DENIED
seven_d_sealed_status                     NOT_OPENED
eight_d_sealed_status                     NOT_OPENED
workflow_status                    COMPLETE_DEV_FAIL
final_qualification_status                     NO_GO
nb39_handoff_mode             negative_evidence_only
sealed_opened                                  False
dtype: object

**STOP:** Development complete. Do not open Sealed without explicit authorization.

## 12. Claim ledger reminder

In [12]:
display(Markdown('''
**Claim ceiling (Route B):** even after eventual Sealed PASS, SBI remains synthetic-domain
qualification under `working_prior_box` — not a real-EEG personalized posterior.

**NB38 scope:** no new *training/calibration* simulations in this notebook.
Notebooks 39–41 may still run their pre-registered local validation / held-out PPC /
control-intervention evaluation simulations when unblocked.
'''))



**Claim ceiling (Route B):** even after eventual Sealed PASS, SBI remains synthetic-domain
qualification under `working_prior_box` — not a real-EEG personalized posterior.

**NB38 scope:** no new *training/calibration* simulations in this notebook.
Notebooks 39–41 may still run their pre-registered local validation / held-out PPC /
control-intervention evaluation simulations when unblocked.
